<a href="https://colab.research.google.com/github/NadiaTeta/group1/blob/main/gpt/Tiny_gpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
 import os
import urllib.request
import zipfile

# 1. URL of the direct raw zip file
url = "https://github.com/saradhix/kinnews_kirnews/raw/master/KINNEWS.zip"
zip_path = "KINNEWS.zip"
extract_dir = "./kinnews_data"

# 2. Download the ZIP file if it doesn't already exist
if not os.path.exists(zip_path):
    print("Downloading KINNEWS dataset...")
    urllib.request.urlretrieve(url, zip_path)

# 3. Extract the contents
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print("Dataset downloaded and extracted successfully!")

Dataset downloaded and extracted successfully!


In [10]:
import pandas as pd

# 1. Define paths to raw CSV files
train_file = os.path.join(extract_dir, "KINNEWS", "raw", "train.csv")
test_file = os.path.join(extract_dir, "KINNEWS", "raw", "test.csv")

# 2. Read the CSV files
df_train = pd.read_csv(train_file)
df_test = pd.read_csv(test_file)

# 3. Combine both dataframes into one
df_full = pd.concat([df_train, df_test], ignore_index=True)

# 4. Keep only the 'content' column
articles = df_full['content']

# 5. Drop empty/null values and removes exact duplicate articles
articles = articles.dropna().drop_duplicates()

print(f"Total unique articles remaining: {len(articles)}")  # Should be ~11,122

Total unique articles remaining: 11122


In [11]:
import re

def clean_text(text):
    # 1. Convert curly quotes/apostrophes to straight ones
    text = text.replace('“', '"').replace('”', '"')
    text = text.replace('‘', "'").replace('’', "'")

    # 2. Keep only alphanumeric characters, spaces, and basic punctuation
    # Permitted characters: a-z, A-Z, 0-9, and basic punctuation marks
    allowed_pattern = r'[^a-zA-Z0-9\s.,!?"\'\(\)\-–:]'
    text = re.sub(allowed_pattern, '', text)

    # 3. Collapse multiple spaces or weird whitespaces into single spaces
    text = re.sub(r'[ \t]+', ' ', text)
    return text.strip()

# Apply cleaning to all extracted articles
cleaned_articles = [clean_text(str(doc)) for doc in articles]

In [12]:
# Join all articles with a double newline as a separator
full_text = "\n\n".join(cleaned_articles)

print(f"Total character count: {len(full_text):,}")  # Should be ~27,000,000 characters

Total character count: 26,850,469


In [13]:
# Calculate the 90% split index
split_idx = int(len(full_text) * 0.9)

# Split into train and validation sets
train_data = full_text[:split_idx]
val_data = full_text[split_idx:]

# Save datasets to local files
with open("train.txt", "w", encoding="utf-8") as f:
    f.write(train_data)

with open("val.txt", "w", encoding="utf-8") as f:
    f.write(val_data)

print(f"Train dataset character length: {len(train_data):,}")
print(f"Validation dataset character length: {len(val_data):,}")

Train dataset character length: 24,165,422
Validation dataset character length: 2,685,047


In [14]:
import torch

# 1. Read the training text to extract the unique vocabulary
with open("train.txt", "r", encoding="utf-8") as f:
    text = f.read()

# 2. Get all unique characters and sort them
chars = sorted(list(set(text)))
vocab_size = len(chars)

# 3. Create mapping dictionaries: character to integer (stoi) and integer to character (itos)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

# 4. Define encode and decode functions
encode = lambda s: [stoi[c] for c in s if c in stoi]             # string -> list of integers
decode = lambda l: ''.join([itos[i] for i in l if i in itos])    # list of integers -> string

print(f"Vocabulary Size: {vocab_size} unique characters")
print(f"Unique characters in dataset:\n{''.join(chars)}")

Vocabulary Size: 78 unique characters
Unique characters in dataset:

 !"'(),-.0123456789:?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz  – 


In [15]:
from torch.utils.data import Dataset, DataLoader

# 1. Load train and validation data
with open("train.txt", "r", encoding="utf-8") as f:
    train_text = f.read()

with open("val.txt", "r", encoding="utf-8") as f:
    val_text = f.read()

# 2. Encode whole datasets into PyTorch integer tensors
train_data = torch.tensor(encode(train_text), dtype=torch.long)
val_data = torch.tensor(encode(val_text), dtype=torch.long)

# 3. Custom PyTorch Dataset to yield training blocks
class TextDataset(Dataset):
    def __init__(self, data, block_size):
        self.data = data
        self.block_size = block_size

    def __len__(self):
        return len(self.data) - self.block_size

    def __getitem__(self, idx):
        # x is the input sequence of length `block_size`
        # y is the target sequence shifted by 1 character to the right
        x = self.data[idx : idx + self.block_size]
        y = self.data[idx + 1 : idx + self.block_size + 1]
        return x, y

# Set context window size (block size)
block_size = 64  # Maximum context length the model looks back to predict the next character

train_dataset = TextDataset(train_data, block_size)
val_dataset = TextDataset(val_data, block_size)

print(f"Train Dataset Samples: {len(train_dataset):,}")
print(f"Validation Dataset Samples: {len(val_dataset):,}")

Train Dataset Samples: 24,163,181
Validation Dataset Samples: 2,684,983


In [16]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# 1. Define dataset dimensions directly
vocab_size = 78
block_size = 64

# Hyperparameters
n_embd = 128     # Embedding dimension
n_head = 4       # Number of attention heads
n_layer = 4      # Number of Transformer blocks
dropout = 0.1    # Dropout rate
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# 1. Single Head Self-Attention
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)

        wei = q @ k.transpose(-2, -1) * (C ** -0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)

        v = self.value(x)
        out = wei @ v
        return out

# 2. Multi-Head Attention
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))

# 3. Feed-Forward Network
class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

# 4. Transformer Block
class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

# 5. Complete GPT Model
class TinyGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens, temperature=0.7, top_k=5):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature

            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')

            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

model = TinyGPT().to(device)
print("Model initialized successfully!")

Model initialized successfully!


In [17]:
from torch.utils.data import DataLoader

batch_size = 64
learning_rate = 1e-3
epochs = 1

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

model.train()
print("Starting Training...")

for i, (x, y) in enumerate(train_loader):
    x, y = x.to(device), y.to(device)

    # Forward pass
    logits, loss = model(x, y)

    # Backward pass & update
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if i % 1000 == 0:
        print(f"Step {i} | Loss: {loss.item():.4f}")

    if i == 5000: # Stop early for a quick demo run
        break

print("Training finished!")

Starting Training...
Step 0 | Loss: 4.5179
Step 1000 | Loss: 1.7876
Step 2000 | Loss: 1.5967
Step 3000 | Loss: 1.4744
Step 4000 | Loss: 1.4871
Step 5000 | Loss: 1.3997
Training finished!


In [20]:
model.eval()

# 1. Define prompt starting with "Ikipe "
context = torch.tensor([encode("Ikipe ")], dtype=torch.long, device=device)

# 2. Generate 200 characters using top-k sampling
generated_ids = model.generate(context, max_new_tokens=200, temperature=0.7, top_k=5)[0].tolist()

print("\n--- GENERATED CLEAN KINYARWANDA TEXT ---")
print(decode(generated_ids))


--- GENERATED CLEAN KINYARWANDA TEXT ---
Ikipe ye ya Afurika y'ingendo z'ibikorwa n'igitabo gitero cya mbere na Maria Commosiaria waba ari uwarambutse abaturage bagatungurambwa umwana wa mbere mu baturage bakaba batabiriye uko bakurikije ibihugu b
